# Random Forest + Cross-Validation — Breast Cancer Classification

**Model:** Random Forest (sklearn ensemble)
**Key concepts:** Ensemble learning, 5-fold stratified CV, feature importance, learning curves

Random Forest improves on single decision trees by averaging many trees trained on random subsets of data and features, reducing variance (overfitting).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, learning_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay
print("Ready.")

## 1. Load Data

In [ ]:
raw = load_breast_cancer(as_frame=True)
X, y = raw.data, raw.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 2. Cross-Validation

Cross-validation gives a more reliable performance estimate than a single train/test split. Here we use **5-fold stratified CV** to ensure each fold has the same class proportion.

We measure three metrics simultaneously: ROC-AUC, accuracy, and F1.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(rf, X, y, cv=cv,
                             scoring=['roc_auc', 'accuracy', 'f1'],
                             return_train_score=True)

print(f"{'Metric':<14} {'Train':>8} {'CV Mean':>10} {'CV Std':>10}")
print('-' * 45)
for m in ['roc_auc', 'accuracy', 'f1']:
    tr  = cv_results[f'train_{m}'].mean()
    val = cv_results[f'test_{m}'].mean()
    std = cv_results[f'test_{m}'].std()
    print(f"{m:<14} {tr:>8.3f} {val:>10.3f} {std:>10.3f}")

## 3. Final Fit and Test Evaluation

In [ ]:
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=raw.target_names))

## 4. Feature Importances

Random Forest provides built-in feature importance scores based on mean decrease in impurity (Gini). Higher = more useful for splitting decisions.

In [ ]:
importance_df = (pd.DataFrame({
    'feature': raw.feature_names,
    'importance': rf.feature_importances_
})
.sort_values('importance', ascending=False)
.head(15))

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(importance_df['feature'][::-1], importance_df['importance'][::-1],
        color='#3498db', edgecolor='white')
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=13)
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

## 5. Learning Curve

A learning curve shows how training and validation performance change as we add more training data.

- **Converging curves** = good generalization
- **Large gap** between train and val = overfitting

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    rf, X, y, cv=cv, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#2ecc71', label='Train')
ax.fill_between(train_sizes,
                train_scores.mean(1) - train_scores.std(1),
                train_scores.mean(1) + train_scores.std(1),
                alpha=0.2, color='#2ecc71')
ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#e74c3c', label='Cross-val')
ax.fill_between(train_sizes,
                val_scores.mean(1) - val_scores.std(1),
                val_scores.mean(1) + val_scores.std(1),
                alpha=0.2, color='#e74c3c')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('ROC-AUC')
ax.set_title('Learning Curve — Random Forest')
ax.legend()
ax.set_ylim(0.9, 1.01)
plt.tight_layout()
plt.show()

## Summary

- Random Forest matches or slightly exceeds Logistic Regression, with ~0.998 CV ROC-AUC.
- **Top features** (`worst perimeter`, `worst concave points`, `worst radius`) align with EDA.
- The learning curve shows train and validation scores converge — no significant overfitting.
- **Next:** Handling class imbalance — a realistic challenge in healthcare data.